In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [6]:
df.shape

(5000, 9)

In [7]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [8]:
categorical = ['lead_source','industry','employment_status','location']
numerical = ['annual_income','number_of_courses_viewed','interaction_count','lead_score']
df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0)

In [9]:
df['industry'].mode()

0    technology
Name: industry, dtype: str

In [10]:
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [14]:
from sklearn.model_selection import train_test_split
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state = 42)
df_train, df_val = train_test_split(df_full_train, test_size = 0.25, random_state = 42)


y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values


del df_train['converted']
del df_val['converted']
del df_test['converted']

In [15]:
from sklearn.metrics import mutual_info_score

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train['converted'])
mi = df_full_train[categorical].apply(mutual_info_converted_score)
print(mi.sort_values(ascending = False).round(2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


In [16]:
from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train[categorical + numerical].to_dict(orient = "records")

dv = DictVectorizer(sparse = False)

X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient = "records")
X_val = dv.transform(val_dicts)

dv.get_feature_names_out()

array(['annual_income', 'employment_status=NA',
       'employment_status=employed', 'employment_status=self_employed',
       'employment_status=student', 'employment_status=unemployed',
       'industry=NA', 'industry=education', 'industry=finance',
       'industry=healthcare', 'industry=manufacturing', 'industry=retail',
       'industry=technology', 'interaction_count', 'lead_score',
       'lead_source=NA', 'lead_source=events',
       'lead_source=organic_search', 'lead_source=paid_ads',
       'lead_source=referral', 'lead_source=social_media', 'location=NA',
       'location=africa', 'location=asia', 'location=europe',
       'location=north_america', 'location=south_america',
       'number_of_courses_viewed'], dtype=object)

In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print(model.intercept_[0])
print(model.coef_.round(3))

y_pred = model.predict_proba(X_val)[:,1]
converted_decision = (y_pred >= 0.5)

og_accuracy = (y_val == converted_decision).mean()
og_accuracy

-0.010838978497809288
[[-0.    -0.001  0.007 -0.003 -0.008 -0.006 -0.    -0.003 -0.001 -0.002
  -0.002 -0.002 -0.001  0.137  0.011  0.    -0.009  0.007 -0.014  0.009
  -0.004  0.    -0.002 -0.002 -0.001 -0.003 -0.004  0.054]]


np.float64(0.645)

In [18]:
def least_useful_feat(df):
    diff_accuracy_dict = {}
    for col in categorical + numerical:
        train_cols = [c for c in categorical + numerical if c!= col]
        train_dicts = df[train_cols].to_dict(orient = "records")
        dv = DictVectorizer(sparse = False)
        X_train = dv.fit_transform(train_dicts)
        val_dicts = df_val[train_cols].to_dict(orient = "records")
        X_val = dv.transform(val_dicts)

        model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:,1]
        converted_decision = (y_pred >= 0.5)

        accuracy = (y_val == converted_decision).mean()
        diff_accuracy_dict[col] =  (accuracy, og_accuracy - accuracy)
    return diff_accuracy_dict

In [19]:
d1 = least_useful_feat(df_train)
d1

{'lead_source': (np.float64(0.642), np.float64(0.0030000000000000027)),
 'industry': (np.float64(0.645), np.float64(0.0)),
 'employment_status': (np.float64(0.645), np.float64(0.0)),
 'location': (np.float64(0.645), np.float64(0.0)),
 'annual_income': (np.float64(0.724), np.float64(-0.07899999999999996)),
 'number_of_courses_viewed': (np.float64(0.643),
  np.float64(0.0020000000000000018)),
 'interaction_count': (np.float64(0.601), np.float64(0.04400000000000004)),
 'lead_score': (np.float64(0.644), np.float64(0.0010000000000000009))}

In [20]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:,1]
    converted_decision = (y_pred >= 0.5)

    accuracy = (y_val == converted_decision).mean().round(3)
    print(c,"\t",accuracy)

1e-06 	 0.598
1e-05 	 0.598
0.0001 	 0.613
0.001 	 0.645
